# Fine-tune Qwen2-VL-2B-Instruct dengan QLoRA

Notebook untuk fine-tune model vision-language pada dataset custom (gambar + pertanyaan -> jawaban).

**Kenapa QLoRA?** VRAM 8.6 GB tidak cukup untuk full fine-tune model 2B
(butuh ~30 GB untuk bobot + gradien + optimizer state). QLoRA me-load base model
dalam 4-bit dan hanya melatih adapter LoRA kecil (~10-20 juta parameter), muat di ~6 GB.

**Urutan pakai:**
1. Jalankan cell 1-2 (config + cek GPU)
2. Siapkan dataset (section 3) - atau generate dummy dulu untuk uji pipeline
3. Jalankan sampai section 8 dengan `SMOKE_TEST = True`
4. Kalau lancar, set `SMOKE_TEST = False` dan training penuh

## 1. Konfigurasi

Semua tombol yang perlu kamu putar ada di sini.

In [1]:
from pathlib import Path
import torch

# ---------- Model ----------
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# ---------- Path ----------
BASE_DIR    = Path.cwd()                      # folder "AI Services"
# Dataset hasil generate_dataset.py: images/, ground_truth/, train.jsonl, val.jsonl
DATA_DIR    = BASE_DIR / "dataset_generator" / "synthetic_furniture_dataset_v2"
TRAIN_FILE  = DATA_DIR / "train.jsonl"
VAL_FILE    = DATA_DIR / "val.jsonl"
GT_DIR      = DATA_DIR / "ground_truth"       # 1 file JSON per sampel (label mentah)
OUTPUT_DIR  = BASE_DIR / "outputs" / "qwen2vl-2b-nesto-lora"

# ---------- Resolusi gambar (PALING menentukan pemakaian VRAM) ----------
# 1 token visual = patch 28x28. max_pixels 768*28*28 -> maksimal ~768 token per gambar.
# Kalau OOM: turunkan MAX_PIXELS ke 512*28*28 atau 384*28*28.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28

# ---------- LoRA ----------
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05
TRAIN_VISION_TOWER = False   # False = vision encoder dibekukan (hemat & biasanya cukup)

# ---------- Training ----------
SMOKE_TEST     = False        # True = latih SMOKE_STEPS step saja untuk cek pipeline
SMOKE_STEPS    = 30          # 5 step terlalu sedikit untuk melihat tren loss sama sekali
EPOCHS         = 3
LR             = 2e-4        # LoRA pada model 2B tahan LR segini; 1e-4 turunnya lambat
BATCH_SIZE     = 2           # per device; naikkan hanya kalau VRAM sisa banyak
GRAD_ACCUM     = 8           # effective batch = BATCH_SIZE * GRAD_ACCUM
WARMUP_RATIO   = 0.03        # rasio, bukan step tetap -- lihat catatan di section 7
MAX_GRAD_NORM  = 1.0         # 0.3 mencekik update di awal (grad norm awal biasanya 2-5)
MAX_SEQ_LEN    = 2048        # sampel lebih panjang dari ini akan dibuang
SEED           = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert TRAIN_FILE.exists(), f"Dataset tidak ditemukan: {TRAIN_FILE} (jalankan dataset_generator/generate_dataset.py)"
print("Config OK ->", OUTPUT_DIR)
print("Dataset   ->", DATA_DIR)

Config OK -> d:\Codelabs\TukangKayu\Nesto.ai\AI Services\outputs\qwen2vl-2b-nesto-lora
Dataset   -> d:\Codelabs\TukangKayu\Nesto.ai\AI Services\dataset_generator\synthetic_furniture_dataset_v2


## 2. Cek GPU

In [2]:
print(f"Torch        : {torch.__version__}")
print(f"CUDA tersedia: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemExit("CUDA tidak terdeteksi. Fine-tuning di CPU tidak realistis untuk model ini.")

props = torch.cuda.get_device_properties(0)
print(f"GPU          : {props.name}")
print(f"VRAM total   : {props.total_memory / 1e9:.1f} GB")

def vram(tag=""):
    """Helper: cetak pemakaian VRAM saat ini."""
    alloc = torch.cuda.memory_allocated() / 1e9
    peak  = torch.cuda.max_memory_allocated() / 1e9
    print(f"[VRAM {tag}] terpakai {alloc:.2f} GB | puncak {peak:.2f} GB")

vram("awal")

Torch        : 2.9.1+cu130
CUDA tersedia: True
GPU          : NVIDIA GeForce RTX 4070 Laptop GPU
VRAM total   : 8.6 GB
[VRAM awal] terpakai 0.00 GB | puncak 0.00 GB


## 3. Dataset

Dataset asli ada di `dataset_generator/synthetic_furniture_dataset/`:

```
synthetic_furniture_dataset/
  images/          1500 sketsa teknis (synth_0001.png ...)
  ground_truth/    1500 label JSON mentah (synth_0001.json ...)
  train.jsonl      1200 sampel
  val.jsonl         300 sampel
```

Task: baca sketsa furnitur -> keluarkan spesifikasi sebagai JSON ketat.
Satu baris `.jsonl` per sampel, path gambar **relatif terhadap `DATA_DIR`**:

```json
{"id": "synth_1064", "messages": [
  {"role": "user", "content": [
      {"type": "image", "image": "images/synth_1064.png"},
      {"type": "text",  "text": "Extract all furniture dimensions, ... into strict JSON."}]},
  {"role": "assistant", "content": "{\"furniture_type\":\"reception_desk\", ...}"}
]}
```

Catatan:
- Jawaban assistant di sini berupa **string** (bukan list part); loader di bawah menormalkannya.
- `ground_truth/<id>.json` adalah isi jawaban yang sama dalam bentuk file terpisah - berguna
  untuk evaluasi per-field (mis. cocokkan `overall_dimensions` hasil prediksi vs label).
- Field `id`, `layout`, `decimal_comma` diabaikan saat training (hanya `messages` yang dipakai).

### 3a. (Opsional) Generate dataset dummy

**Dataset asli sudah tersedia** di `dataset_generator/synthetic_furniture_dataset/`
(1200 train / 300 val + 1500 gambar). Cell ini dibiarkan `GENERATE_DUMMY = False`
karena kalau dijalankan ia akan **menimpa `train.jsonl` dan `val.jsonl` asli**.

Hanya nyalakan kalau kamu sengaja ingin menguji pipeline dengan data mainan,
dan arahkan `DATA_DIR` ke folder lain dulu.

In [3]:
from PIL import Image, ImageDraw
import json, random

GENERATE_DUMMY = False   # dataset asli sudah ada -- JANGAN diubah ke True (menimpa train.jsonl)

if GENERATE_DUMMY:
    random.seed(SEED)
    img_dir = DATA_DIR / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    SHAPES = {
        "persegi":   lambda d: d.rectangle([60, 60, 260, 260], fill=(150, 100, 60)),
        "lingkaran": lambda d: d.ellipse([60, 60, 260, 260], fill=(150, 100, 60)),
        "segitiga":  lambda d: d.polygon([(160, 50), (270, 270), (50, 270)], fill=(150, 100, 60)),
    }

    rows = []
    for i in range(24):
        name = random.choice(list(SHAPES))
        img = Image.new("RGB", (320, 320), (240, 230, 210))
        SHAPES[name](ImageDraw.Draw(img))
        rel = f"images/dummy_{i:03d}.png"
        img.save(DATA_DIR / rel)
        rows.append({"messages": [
            {"role": "user", "content": [
                {"type": "image", "image": rel},
                {"type": "text", "text": "Bentuk apa yang ada di gambar ini?"}]},
            {"role": "assistant", "content": [
                {"type": "text", "text": f"Bentuk pada gambar adalah {name}."}]},
        ]})

    random.shuffle(rows)
    split = int(len(rows) * 0.85)
    for path, subset in [(TRAIN_FILE, rows[:split]), (VAL_FILE, rows[split:])]:
        with open(path, "w", encoding="utf-8") as f:
            for r in subset:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Dummy dataset dibuat: {split} train / {len(rows) - split} val")
else:
    print("Dilewati - pakai dataset asli")

Dilewati - pakai dataset asli


### 3b. Load dataset

In [4]:
import json
from PIL import Image
from torch.utils.data import Dataset


class VLJsonlDataset(Dataset):
    """Baca .jsonl berisi percakapan, resolve path gambar jadi objek PIL saat diakses.

    Gambar sengaja TIDAK dimuat di __init__ supaya RAM tidak meledak untuk dataset besar.
    """

    def __init__(self, path, image_root, min_pixels, max_pixels):
        self.rows = []
        with open(path, encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    self.rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f"{path} baris {line_no} bukan JSON valid: {e}") from e
        self.image_root = Path(image_root)
        self.min_pixels = min_pixels
        self.max_pixels = max_pixels

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        messages = json.loads(json.dumps(self.rows[idx]))  # deep copy, jangan mutasi cache
        for msg in messages["messages"]:
            content = msg["content"]
            if isinstance(content, str):
                msg["content"] = [{"type": "text", "text": content}]
                continue
            for part in content:
                if part.get("type") == "image" and isinstance(part["image"], str):
                    img_path = self.image_root / part["image"]
                    if not img_path.exists():
                        raise FileNotFoundError(f"Gambar tidak ditemukan: {img_path}")
                    part["image"] = Image.open(img_path).convert("RGB")
                    # dibaca oleh qwen_vl_utils untuk membatasi jumlah token visual
                    part["min_pixels"] = self.min_pixels
                    part["max_pixels"] = self.max_pixels
        return messages


train_ds = VLJsonlDataset(TRAIN_FILE, DATA_DIR, MIN_PIXELS, MAX_PIXELS)
val_ds   = VLJsonlDataset(VAL_FILE,   DATA_DIR, MIN_PIXELS, MAX_PIXELS)

print(f"Train: {len(train_ds)} sampel")
print(f"Val  : {len(val_ds)} sampel")
print("\nContoh sampel pertama:")
print(json.dumps(train_ds.rows[0], indent=2, ensure_ascii=False)[:600])

Train: 1200 sampel
Val  : 300 sampel

Contoh sampel pertama:
{
  "id": "synth_1216",
  "layout": "isometric",
  "decimal_comma": true,
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "image",
          "image": "images/synth_1216.png"
        },
        {
          "type": "text",
          "text": "Extract all furniture dimensions, partitions, and structural specs from this technical sketch into strict JSON."
        }
      ]
    },
    {
      "role": "assistant",
      "content": "{\"furniture_type\":\"base_cabinet\",\"overall_dimensions\":{\"length_cm\":108.9,\"width_cm\":58.1,\"height_cm\":86.5},\"plinth\


## 4. Load processor & model 4-bit

`prepare_model_for_kbit_training` melakukan beberapa hal penting: meng-cast layer norm ke fp32
(stabilitas numerik), menyalakan gradient checkpointing, dan memastikan input embedding
menghasilkan gradien meski bobotnya beku.

In [5]:
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = "right"   # training pakai padding kanan

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)

model.config.use_cache = False   # tidak kompatibel dengan gradient checkpointing
vram("setelah load model 4-bit")

d:\Codelabs\TukangKayu\Nesto.ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 729/729 [00:04<00:00, 151.60it/s]


[VRAM setelah load model 4-bit] terpakai 1.52 GB | puncak 1.57 GB


In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model.enable_input_require_grads()

TEXT_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_kwargs = dict(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TEXT_MODULES,
)
if not TRAIN_VISION_TOWER:
    # vision tower Qwen2-VL bernama "visual.*" -- dikecualikan agar tetap beku
    lora_kwargs["exclude_modules"] = r".*visual.*"

model = get_peft_model(model, LoraConfig(**lora_kwargs))
model.print_trainable_parameters()
vram("setelah pasang LoRA")

trainable params: 36,929,536 || all params: 2,245,915,136 || trainable%: 1.6443
[VRAM setelah pasang LoRA] terpakai 2.15 GB | puncak 2.46 GB


## 5. Collator: bagian tersulit

Dua hal yang harus benar di sini, dan keduanya adalah sumber bug paling umum:

1. **Batching multimodal.** `apply_chat_template` menghasilkan teks dengan placeholder
   `<|image_pad|>`; processor mengembangkannya jadi N token sesuai ukuran gambar setelah
   *smart resize*. Daftar gambar harus **datar** (flat) dan urut sesuai kemunculannya di batch.

2. **Masking label.** Loss hanya boleh dihitung pada token jawaban assistant. Semua sisanya
   (system prompt, pertanyaan user, token gambar, padding) di-set `-100` supaya diabaikan
   `CrossEntropyLoss`. Kalau ini salah, model belajar menghafal pertanyaan - atau loss langsung 0.

Masking di bawah bekerja dengan memindai penanda `<|im_start|>assistant` ... `<|im_end|>`,
jadi otomatis benar untuk percakapan single-turn maupun multi-turn.

In [7]:
from qwen_vl_utils import process_vision_info

tok = processor.tokenizer

IM_START_ID   = tok.convert_tokens_to_ids("<|im_start|>")
IM_END_ID     = tok.convert_tokens_to_ids("<|im_end|>")
ASSISTANT_IDS = tok.encode("assistant\n", add_special_tokens=False)
PAD_ID        = tok.pad_token_id

assert None not in (IM_START_ID, IM_END_ID), "Token penanda chat tidak ditemukan"


def build_labels(input_ids):
    """Kembalikan label dengan -100 di mana pun kecuali isi giliran assistant."""
    labels = torch.full_like(input_ids, -100)
    n_assist = len(ASSISTANT_IDS)

    for row in range(input_ids.size(0)):
        ids = input_ids[row]
        i = 0
        while i < ids.size(0):
            if ids[i] == IM_START_ID and ids[i + 1 : i + 1 + n_assist].tolist() == ASSISTANT_IDS:
                start = i + 1 + n_assist
                end = start
                while end < ids.size(0) and ids[end] != IM_END_ID:
                    end += 1
                stop = min(end + 1, ids.size(0))   # sertakan <|im_end|> agar model belajar berhenti
                labels[row, start:stop] = ids[start:stop]
                i = stop
            else:
                i += 1

    labels[input_ids == PAD_ID] = -100
    return labels


class QwenVLCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts, images = [], []
        for ex in examples:
            msgs = ex["messages"]
            texts.append(self.processor.apply_chat_template(msgs, tokenize=False))
            img, _vid = process_vision_info(msgs)
            if img:
                images.extend(img)          # flat, bukan nested

        batch = self.processor(
            text=texts,
            images=images or None,
            padding=True,
            return_tensors="pt",
        )
        batch["labels"] = build_labels(batch["input_ids"])
        return batch


collator = QwenVLCollator(processor)
print("Collator siap")

Collator siap


## 6. Sanity check collator

Verifikasi sebelum training, bukan sesudah. Yang dicek:
- token yang **dilatih** hanya jawaban assistant
- panjang sekuens masih wajar (kalau ribuan token, turunkan `MAX_PIXELS`)

In [8]:
batch = collator([train_ds[0], train_ds[1]])

print("Bentuk tensor:")
for k, v in batch.items():
    print(f"  {k:20s} {tuple(v.shape)}")

seq_len = batch["input_ids"].shape[1]
n_supervised = int((batch["labels"] != -100).sum())
print(f"\nPanjang sekuens : {seq_len} token")
print(f"Token dilatih   : {n_supervised} dari {batch['labels'].numel()}")

assert n_supervised > 0, "TIDAK ADA token yang dilatih - masking label salah!"

print("\n--- Yang dipelajari model (label != -100) ---")
kept = batch["labels"][0]
print(repr(tok.decode(kept[kept != -100])))

print("\n--- Konteks penuh sampel 0, token gambar diringkas jadi @ ---")
print(tok.decode(batch["input_ids"][0])[:800].replace("<|image_pad|>", "@"))

Bentuk tensor:
  input_ids            (2, 908)
  attention_mask       (2, 908)
  mm_token_type_ids    (2, 908)
  pixel_values         (4992, 1176)
  image_grid_thw       (2, 3)
  labels               (2, 908)

Panjang sekuens : 908 token
Token dilatih   : 279 dari 1816

--- Yang dipelajari model (label != -100) ---
'{"furniture_type":"base_cabinet","overall_dimensions":{"length_cm":108.9,"width_cm":58.1,"height_cm":86.5},"plinth":{"has_plinth":true,"height_cm":8.2,"offset_cm":3.0},"partitions":{"shelves_count":1,"doors_count":1,"drawers_count":1},"material":{"board_material":"Blockboard","board_thickness_mm":25,"finish":"Veneer","finish_color":"Hitam","finish_code":null},"drop_pocket":null,"has_curve":false,"default_thickness_mm":18,"default_edging_mm":1.0}<|im_end|>'

--- Konteks penuh sampel 0, token gambar diringkas jadi @ ---
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|>@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@<|image


In [9]:
# Buang sampel yang terlalu panjang supaya tidak OOM di tengah training.
def filter_too_long(ds, max_len):
    keep = []
    for i in range(len(ds)):
        n = collator([ds[i]])["input_ids"].shape[1]
        if n <= max_len:
            keep.append(ds.rows[i])
        else:
            print(f"  dibuang idx {i}: {n} token > {max_len}")
    ds.rows = keep
    return ds

print(f"Menyaring sampel > {MAX_SEQ_LEN} token...")
train_ds = filter_too_long(train_ds, MAX_SEQ_LEN)
val_ds   = filter_too_long(val_ds,   MAX_SEQ_LEN)
print(f"Sisa: {len(train_ds)} train / {len(val_ds)} val")

Menyaring sampel > 2048 token...
Sisa: 1200 train / 300 val


## 7. Trainer

Dipakai `Trainer` bawaan `transformers`, bukan `SFTTrainer`. Alasannya: dengan collator custom
seperti di atas, TRL tetap harus di-set `skip_prepare_dataset=True` sehingga tidak memberi nilai
tambah, sementara signature `SFTConfig` cukup sering berubah antar versi.

**Warmup dipakai sebagai rasio, bukan jumlah step tetap.** Dengan `warmup_steps=10` di smoke
test 5 step, learning rate tidak pernah sampai nilai penuh - di step 5 baru separuh - sehingga
loss terlihat "macet" padahal pipeline-nya benar. `warmup_steps=0.03` selalu proporsional:
~0 step saat smoke test (langsung LR penuh, jadi tren terlihat) dan ~13 step di training penuh.

`max_grad_norm` sengaja dinaikkan dari 0.3 ke 1.0. Nilai 0.3 berasal dari setelan paper QLoRA
untuk model besar; di sini grad norm awal ada di kisaran 2-5, jadi 0.3 memperkecil tiap update
sekitar 5-10x dan membuat penurunan loss awal jauh lebih lambat dari seharusnya.

`paged_adamw_8bit` menjaga optimizer state tetap kecil dan memindahkannya ke RAM saat VRAM
menipis - penting di GPU 8 GB.

In [10]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE_TEST else -1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_RATIO,  # transformers v5: float < 1 = rasio dari total step
    optim="paged_adamw_8bit",
    bf16=True,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=1 if SMOKE_TEST else 5,
    eval_strategy="no" if SMOKE_TEST else "epoch",
    save_strategy="no" if SMOKE_TEST else "epoch",
    save_total_limit=2,
    remove_unused_columns=False,   # dataset kita bukan datasets.Dataset
    dataloader_num_workers=0,      # Windows: >0 sering bermasalah dengan objek PIL
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=None if SMOKE_TEST else val_ds,
    data_collator=collator,
)

print("Mode:", f"SMOKE TEST ({SMOKE_STEPS} step)" if SMOKE_TEST else "TRAINING PENUH")

Mode: TRAINING PENUH


## 8. Training

**Smoke test dulu.** Yang harus kamu lihat:
- tidak OOM
- `loss` keluar angka wajar (umumnya 1-5 di awal), **bukan** 0.0 dan **bukan** `nan`

Loss 0.0 = masking label salah (semua ter-mask). `nan` = masalah numerik, coba turunkan LR.

**Cara membaca angka loss di task ini.** Target-nya ~180 token JSON, dan sebagian besar isinya
adalah angka yang hanya bisa dibaca dari gambar (`177.4`, `65.8`, ...). Angka itu tidak bisa
ditebak dari pola bahasa, jadi ada *lantai* loss yang tidak akan pernah jadi ~0.
Yang harus turun cepat adalah bagian strukturnya (nama field, kurung, koma).

Patokan realistis dengan setelan sekarang (RTX 4070 Laptop 8.6 GB, VRAM puncak ~3.5 GB,
~20 detik per optimizer step):

| Titik | Loss wajar |
|---|---|
| step 1-5    | 1.8 - 2.2 (belum belajar apa-apa, ini normal) |
| step ~30    | 0.8 - 1.2 (struktur JSON mulai dikuasai) |
| akhir epoch 1 | 0.3 - 0.6 |
| akhir epoch 3 | 0.15 - 0.35 |

**Jangan menilai apa pun dari 5 step.** 5 step x `GRAD_ACCUM` 8 = 40 dari 1200 sampel, yaitu 3%
dari satu epoch. Karena itu `SMOKE_STEPS` sekarang 30 - cukup untuk melihat tren, masih ~10 menit.

Karena VRAM masih sisa banyak, kamu bisa menaikkan `MAX_PIXELS` ke `1024 * 28 * 28`
atau `BATCH_SIZE` ke 2 kalau butuh detail visual lebih tinggi - untuk task yang harus membaca
angka dari sketsa, resolusi lebih tinggi biasanya berpengaruh besar ke akurasi akhir.

Kalau lolos, kembali ke cell config, set `SMOKE_TEST = False`, lalu **restart kernel** dan
jalankan ulang dari atas.

In [11]:
torch.cuda.reset_peak_memory_stats()
result = trainer.train()
vram("puncak training")
print(result.metrics)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.005868,0.004696
2,0.001880,0.002403
3,0.001018,0.002277


[VRAM puncak training] terpakai 2.19 GB | puncak 9.31 GB
{'train_runtime': 12587.4886, 'train_samples_per_second': 0.286, 'train_steps_per_second': 0.018, 'total_flos': 4.5246560416512e+16, 'train_loss': 0.0491767905631827, 'epoch': 3.0}


## 9. Simpan adapter

Hanya adapter LoRA yang disimpan (beberapa puluh MB), bukan bobot penuh 4 GB.

In [12]:
ADAPTER_DIR = OUTPUT_DIR / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Adapter tersimpan di {ADAPTER_DIR} ({size_mb:.1f} MB)")

Adapter tersimpan di d:\Codelabs\TukangKayu\Nesto.ai\AI Services\outputs\qwen2vl-2b-nesto-lora\adapter (159.2 MB)


## 10. Inference dengan adapter

Restart kernel dulu untuk membebaskan VRAM, lalu jalankan cell config (section 1),
cek GPU (section 2), dan cell di bawah ini.

In [13]:
import json
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from PIL import Image

ADAPTER_DIR = OUTPUT_DIR / "adapter"

infer_processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS
)
infer_processor.tokenizer.padding_side = "left"   # generate pakai padding kiri

base = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    dtype=torch.bfloat16,
    device_map={"": 0},
)
infer_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
infer_model.eval()


def ask(image_path, question, max_new_tokens=256):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": Image.open(image_path).convert("RGB"),
         "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
        {"type": "text", "text": question},
    ]}]

    text = infer_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, _ = process_vision_info(messages)
    inputs = infer_processor(text=[text], images=images, padding=True, return_tensors="pt")
    inputs = inputs.to(infer_model.device)

    with torch.inference_mode():
        out = infer_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    trimmed = out[0][inputs.input_ids.shape[1]:]
    return infer_processor.decode(trimmed, skip_special_tokens=True)


# contoh: pakai sampel validasi pertama
def as_text(content):
    """Dataset ini menulis jawaban assistant sebagai string biasa; sampel lain bisa list part."""
    if isinstance(content, str):
        return content
    return "".join(p["text"] for p in content if p.get("type") == "text")


sample = json.loads(open(VAL_FILE, encoding="utf-8").readline())
user_parts = sample["messages"][0]["content"]
img_rel  = next(p["image"] for p in user_parts if p["type"] == "image")
question = next(p["text"] for p in user_parts if p["type"] == "text")

print("Pertanyaan   :", question)
print("Jawaban model:", ask(DATA_DIR / img_rel, question))
print("Ground truth :", as_text(sample["messages"][1]["content"]))

Loading weights: 100%|██████████| 729/729 [00:04<00:00, 163.03it/s]


Pertanyaan   : Extract all furniture dimensions, partitions, and structural specs from this technical sketch into strict JSON.
Jawaban model: {"furniture_type":"reception_desk","overall_dimensions":{"length_cm":178.7,"width_cm":54.7,"height_cm":90.2},"plinth":{"has_plinth":true,"height_cm":8.5,"offset_cm":4.6},"partitions":{"shelves_count":0,"doors_count":0,"drawers_count":0},"material":{"board_material":"MDF","board_thickness_mm":12,"finish":"Duco","finish_color":null,"finish_code":null},"drop_pocket":null,"has_curve":true,"default_thickness_mm":18,"default_edging_mm":1.0}
Ground truth : {"furniture_type":"reception_desk","overall_dimensions":{"length_cm":178.7,"width_cm":54.7,"height_cm":90.2},"plinth":{"has_plinth":true,"height_cm":8.5,"offset_cm":4.6},"partitions":{"shelves_count":0,"doors_count":0,"drawers_count":0},"material":{"board_material":"MDF","board_thickness_mm":12,"finish":"Duco","finish_color":null,"finish_code":null},"drop_pocket":null,"has_curve":true,"default_thickne

## 11. Merge adapter ke bobot penuh

Hanya perlu kalau mau deploy tanpa dependensi `peft`. **Merge harus dilakukan di atas model
non-quantized** (bf16) - merge di atas bobot 4-bit menghasilkan model yang rusak.

Butuh ~10 GB RAM (bukan VRAM, karena di-load ke CPU) dan menghasilkan ~4.4 GB di disk.
Logikanya ada di `merge_adapter.py` supaya bisa dijalankan tanpa notebook:

```
uv run python "AI Services/merge_adapter.py"
```

Script itu **tidak menerima `--min-pixels`/`--max-pixels`, dan itu memang disengaja.**
`processor.save_pretrained()` di section 9 sudah menulis batas resolusi training ke
`adapter/processor_config.json` (sebagai `size.shortest_edge`/`longest_edge`, yaitu
MIN/MAX_PIXELS), jadi script menyalin processor dari folder adapter apa adanya. Kalau nilai
itu bisa dioper lewat flag, suatu saat `MAX_PIXELS` di section 1 diubah dan merge dijalankan
dengan nilai lama - hasilnya model yang membaca gambar pada resolusi berbeda dari saat
training, akurasi turun, tanpa satu pun pesan error. Base model juga dibaca dari
`adapter_config.json`, bukan dari `MODEL_ID` di notebook, dengan alasan yang sama.

**Selalu verifikasi setelah merge**, jangan diasumsikan setara adapter:

```
uv run python "AI Services/evaluate.py"   --model-id "AI Services/outputs/qwen2vl-2b-nesto-lora/merged"   --no-adapter --limit 40 --no-downstream
```

Skornya harus sama dengan hasil evaluasi adapter. Kalau anjlok, penyebab paling sering adalah
merge dilakukan di atas base 4-bit (bukan bf16).

In [15]:
MERGE = True   # set False kalau adapter saja sudah cukup

if MERGE:
    import subprocess, sys

    # Tidak ada flag resolusi/model di sini: script membaca keduanya dari folder
    # adapter supaya tidak ada dua sumber kebenaran. Lihat markdown di atas.
    cmd = [
        sys.executable, str(BASE_DIR / "merge_adapter.py"),
        "--adapter", str(OUTPUT_DIR / "adapter"),
        "--out",     str(OUTPUT_DIR / "merged"),
    ]
    # Proses terpisah supaya ~10 GB RAM base bf16 langsung dilepas setelah selesai,
    # bukan menempel di kernel notebook sampai restart.
    subprocess.run(cmd, check=True)
else:
    print("Dilewati")

## Troubleshooting

| Gejala | Penyebab & solusi |
|---|---|
| `CUDA out of memory` | Turunkan `MAX_PIXELS` (768 -> 512 -> 384). Ini paling efektif untuk model VL. Lalu pastikan `BATCH_SIZE=1` dan naikkan `GRAD_ACCUM`. |
| `loss = 0.0` sejak step 1 | Masking label salah - semua token ter-mask. Jalankan ulang sanity check (section 6). |
| `loss = nan` | Turunkan `LR` ke `5e-5`, pastikan `bf16=True` (bukan `fp16`). |
| `Image features and image tokens do not match` | Daftar gambar tidak datar atau tidak urut. Pastikan `images.extend(img)`, bukan `append`. |
| Loss mentok di ~1.8-2.0 dan tidak turun | Cek dulu berapa step yang sudah jalan. Di bawah ~20 step ini normal. Kalau setelah 50+ step masih segitu: pastikan `warmup_steps` diisi rasio (mis. 0.03), bukan jumlah step tetap yang lebih besar dari `max_steps`, `max_grad_norm >= 1.0`, dan `LR = 2e-4`. |
| Training jalan tapi model tidak berubah | LoRA tidak terpasang di modul yang benar. Cek `print_trainable_parameters()` - harus > 0, sekitar 0.5-2% dari total. |
| Sangat lambat | Normal di laptop GPU: ~1-3 detik per step. Gradient checkpointing menukar kecepatan dengan memori. |
| `bitsandbytes` error di Windows | Pastikan versi >= 0.45; jalankan `uv sync` ulang. |

## Setelah training berhasil

1. **Evaluasi kualitatif dulu** - jalankan `ask()` pada 10-20 sampel validasi dan baca hasilnya.
   Angka loss saja menipu untuk task generatif.
2. **Kalau overfit** (loss train turun, val naik): kurangi `EPOCHS` ke 2, turunkan `LORA_R` ke 8,
   atau tambah data.
3. **Kalau underfit** (jawaban masih generik): naikkan `EPOCHS`, `LORA_R` ke 32, atau `LR` ke `2e-4`.
4. **Kalau model buta detail visual** (bahasa benar tapi salah lihat gambar): naikkan `MAX_PIXELS`,
   dan pertimbangkan `TRAIN_VISION_TOWER = True` kalau VRAM cukup.